In [1]:
import numpy as np
import gzip
import pickle

In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pablotab/mnistpklgz")

print("Path to dataset files:", path)

100%|██████████| 32.1M/32.1M [00:12<00:00, 2.65MB/s]

Extracting files...


Path to dataset files: C:\Users\Vivek Tiwari\.cache\kagglehub\datasets\pablotab\mnistpklgz\versions\1


Path to dataset files: C:\Users\Vivek Tiwari\.cache\kagglehub\datasets\pablotab\mnistpklgz\versions\1

In [27]:
# Load dataset (you can use mnist.pkl.gz)
with gzip.open('mnist.pkl.gz', 'rb') as f:
    train_set, valid_set, test_set = pickle.load(f, encoding='latin1')

C:\Users\Vivek Tiwari\AppData\Local\Temp\ipykernel_24896\1730844430.py:3: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  train_set, valid_set, test_set = pickle.load(f, encoding='latin1')


In [28]:
X_train, y_train = train_set
X_test, y_test = test_set

In [29]:
# Normalize (already in 0–1 if using mnist.pkl.gz)
X_train = X_train / 1.0
X_test = X_test / 1.0

In [30]:
# One-hot encoding
def one_hot(y, num_classes=10):
    return np.eye(num_classes)[y]

In [31]:
y_train = one_hot(y_train)
y_test = one_hot(y_test)

In [32]:
def init_params():
    np.random.seed(42)
    params = {
        "W1": np.random.randn(784, 128) * 0.01,
        "b1": np.zeros((1, 128)),
        "W2": np.random.randn(128, 64) * 0.01,
        "b2": np.zeros((1, 64)),
        "W3": np.random.randn(64, 32) * 0.01,
        "b3": np.zeros((1, 32)),
        "W4": np.random.randn(32, 10) * 0.01,
        "b4": np.zeros((1, 10)),
    }
    return params

In [33]:
def relu(Z):
    return np.maximum(0, Z)

def relu_deriv(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return expZ / np.sum(expZ, axis=1, keepdims=True)

In [34]:
def forward(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)

    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = relu(Z2)

    Z3 = A2 @ params["W3"] + params["b3"]
    A3 = relu(Z3)

    Z4 = A3 @ params["W4"] + params["b4"]
    A4 = softmax(Z4)

    cache = (Z1, A1, Z2, A2, Z3, A3, Z4, A4)
    return A4, cache

In [35]:
def compute_loss(Y, Y_hat):
    m = Y.shape[0]
    return -np.sum(Y * np.log(Y_hat + 1e-8)) / m

In [36]:
def backward(X, Y, params, cache):
    Z1, A1, Z2, A2, Z3, A3, Z4, A4 = cache
    m = X.shape[0]

    dZ4 = A4 - Y
    dW4 = A3.T @ dZ4 / m
    db4 = np.sum(dZ4, axis=0, keepdims=True) / m

    dZ3 = (dZ4 @ params["W4"].T) * relu_deriv(Z3)
    dW3 = A2.T @ dZ3 / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    dZ2 = (dZ3 @ params["W3"].T) * relu_deriv(Z2)
    dW2 = A1.T @ dZ2 / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dZ1 = (dZ2 @ params["W2"].T) * relu_deriv(Z1)
    dW1 = X.T @ dZ1 / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1,
             "dW2": dW2, "db2": db2,
             "dW3": dW3, "db3": db3,
             "dW4": dW4, "db4": db4}

    return grads

In [37]:
def update(params, grads, lr):
    for key in params:
        params[key] -= lr * grads["d" + key]
    return params

In [38]:
def train(X, Y, epochs=20, batch_size=64, lr=0.001):
    params = init_params()
    m = X.shape[0]

    for epoch in range(epochs):
        perm = np.random.permutation(m)
        X_shuff = X[perm]
        Y_shuff = Y[perm]

        for i in range(0, m, batch_size):
            X_batch = X_shuff[i:i+batch_size]
            Y_batch = Y_shuff[i:i+batch_size]

            Y_hat, cache = forward(X_batch, params)
            grads = backward(X_batch, Y_batch, params, cache)
            params = update(params, grads, lr)

        # Loss after each epoch
        Y_hat, _ = forward(X, params)
        loss = compute_loss(Y, Y_hat)
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

    return params

In [39]:
def accuracy(X, Y, params):
    Y_hat, _ = forward(X, params)
    preds = np.argmax(Y_hat, axis=1)
    labels = np.argmax(Y, axis=1)
    return np.mean(preds == labels)

# Train
params = train(X_train, y_train)

# Test accuracy
acc = accuracy(X_test, y_test, params)
print("Test Accuracy:", acc)

Epoch 1, Loss: 2.3024
Epoch 2, Loss: 2.3022
Epoch 3, Loss: 2.3020
Epoch 4, Loss: 2.3018
Epoch 5, Loss: 2.3017
Epoch 6, Loss: 2.3016
Epoch 7, Loss: 2.3015
Epoch 8, Loss: 2.3014
Epoch 9, Loss: 2.3014
Epoch 10, Loss: 2.3013
Epoch 11, Loss: 2.3013
Epoch 12, Loss: 2.3012
Epoch 13, Loss: 2.3012
Epoch 14, Loss: 2.3012
Epoch 15, Loss: 2.3012
Epoch 16, Loss: 2.3011
Epoch 17, Loss: 2.3011
Epoch 18, Loss: 2.3011
Epoch 19, Loss: 2.3011
Epoch 20, Loss: 2.3011
Test Accuracy: 0.1135
